    1) Перемешать магнитуды и времена событий
    2) Вычислить значения NND для всех пар событий
    3) Использовать 1-й процентиль как оценку для eta_0
        -> Обратите внимание, что eta_0 необходима для разделения
           кластерных и фоновых событий на следующих этапах анализа
    4) eta_0 сохраняется в директории /data
       имя файла: [имя_файла]_Mc_[порог_магнитуды]_eta_0.txt

In [1]:
import matplotlib as mpl
#mpl.use( 'Agg') # отключить интерактивные графики
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import os

#------------------------------мои модули--------------------------------------
import src.clustering as clustering
import src.data_utils as data_utils
from   src.EqCat import *

In [2]:
# Создание объектов для работы с каталогами землетрясений:

eqCat   = EqCat()       # оригинальный каталог
ranCat  = EqCat()       # рандомизированный (пуассоновский) каталог
eqCatMc = EqCat()       # каталог с событиями выше порога полноты
np.random.seed(123456)  # фиксируем случайные числа для воспроизводимости

In [3]:
# ------------------------------------------------------------------------------------
#  КОНФИГУРАЦИЯ ВХОДНЫХ ДАННЫХ
# ------------------------------------------------------------------------------------
dir_in  = '../data'                          # директория с данными
file_in = 'NEIC_Global_Chignik_2021.mat'  # входной файл с данными о землетрясениях

In [4]:
# Параметры анализа
dPar  = {   
    'aMc'         :  np.array([3.0, 4.0]),   # пороговые значения магнитуд (Mc)
    'D'           :  1.6,                    # TODO: фрактальная размерность определяется на основе данных
    'b'           :  0.5276475013777839,     # b-значение (вычисляется отдельно)
    
    # количество бутстреп-итераций для рандомизированных каталогов
    'nBoot' : 100,
    #=================параметры построения графиков================
    'eta_binsize' :  .3,  # размер ячейки гистограммы
    
    'cmin' : 1, 
    'xmin' : -13, 'xmax' : 0,  # диапазон по оси X для гистограммы

    ## Параметры для графика "время-расстояние" (T-R plot)
    'binx'      :  .1,    'biny' :  .1,     # размер ячеек для плотности и сглаживания
    'sigma'     :  None,                    # ширина полосы Гаусса (если None - автоматически)
    'Tmin'      :  -8,    'Tmax' :  0,      # диапазон по времени
    'Rmin'      :  -5,    'Rmax' :  3,      # диапазон по расстоянию
    'cmap'      :  plt.cm.RdYlGn_r,         # цветовая схема
    'showPlot'  :  False,                   # показывать ли графики (для отладки)
}

In [6]:
# ------------------------------------------------------------------------------------
#  ЗАГРУЗКА ДАННЫХ, ОТБОР СОБЫТИЙ
# ------------------------------------------------------------------------------------
# Загружаем данные из бинарного файла .mat
eqCat.loadMatBin(os.path.join(dir_in, file_in))
print('общее количество событий', eqCat.size())

# Отбираем события с магнитудой выше начального порога
eqCat.selectEvents(dPar['aMc'][0], None, 'Mag')
print('количество событий после начального отбора', eqCat.size())

# Преобразуем географические координаты в декартовы (для вычисления расстояний)
eqCat.toCart_coordinates(projection = 'eqdc')  # эквидистантная проекция

# Цикл по разным порогам магнитуды (Mc)
for f_Mc in dPar['aMc']:
    print( '-------------- текущий порог Mc:', f_Mc, '---------------------')
    
    # Создаем подкаталог только с событиями выше текущего порога Mc
    eqCatMc.copy( eqCat)
    eqCatMc.selectEvents( f_Mc, None, 'Mag')
    print( 'размер каталога после отбора по магнитуде', eqCat.size())
    
    # Словарь констант для использования в модуле кластеризации
    dConst = {'Mc' : f_Mc,
               'b' : dPar['b'],
               'D' : dPar['D']}

    #=============================2===================================================
    #                    рандомизация каталога (бутстреп)
    #=================================================================================
    # Массив для хранения значений eta_0 из каждой итерации
    a_Eta_0 = np.zeros( dPar['nBoot'])
    
    # Цикл по бутстреп-итерациям
    for i_Bs in range( dPar['nBoot']):
        # Копируем оригинальный каталог
        ranCat.copy( eqCatMc)
        
        # Рандомизируем пространственные координаты (X, Y) и время
        ranCat.data['X']     = np.random.uniform( eqCatMc.data['X'].min(), eqCatMc.data['X'].max(), size = eqCatMc.size())
        ranCat.data['Y']     = np.random.uniform( eqCatMc.data['Y'].min(), eqCatMc.data['Y'].max(), size = eqCatMc.size())
        ranCat.data['Time']  = clustering.rand_rate_uni( eqCatMc.size(), eqCatMc.data['Time'].min(), eqCatMc.data['Time'].max())
        ranCat.sortCatalog( 'Time')  # сортируем по времени
        
        #==================================3=============================================
        #     вычисляем расстояния до ближайшего соседа (NND) для рандомизированного каталога
        #================================================================================
        dNND = clustering.NND_eta( ranCat, dConst, M0=0, correct_co_located = True, verbose = False)
        
        # Вычисляем 1-й процентиль логарифмов расстояний - это и есть eta_0 для данной итерации
        a_Eta_0[i_Bs] = round( np.percentile( np.log10(dNND['aNND']), 1), 5)
        # print( 'итерация', i_Bs+1,'из', dPar['nBoot'], 'eta_0 - 1-й процентиль:', np.percentile( np.log10(dNND['aNND']), 1))
        
        # Если включено отображение графиков - показываем промежуточные результаты
        if dPar['showPlot'] == True:
            #=================================4==============================================
            #                          строим гистограмму NND
            #================================================================================
            plt.figure( 1, figsize = (10,5))
            ax = plt.axes( [.12, .12, .83, .83])
            # Строим гистограмму логарифмов расстояний
            ax.hist( np.log10( dNND['aNND']), np.arange( dPar['xmin'], dPar['xmax'], dPar['eta_binsize']),
                            color = '.5', label = 'Mc = %.1f'%( f_Mc), align = 'mid', rwidth=.9)
            
            # Отмечаем характерные значения на графике
            ax.plot( [-5, -5], ax.get_ylim(), 'w-',  lw = 2, )
            ax.plot( [-5, -5], ax.get_ylim(), 'k--', lw = 2, )
            ax.plot( [a_Eta_0[i_Bs], a_Eta_0[i_Bs]], ax.get_ylim(), 'w-',  lw = 2, label = '$N_\mathrm{tot}$=%i'%( ranCat.size()))
            ax.plot( [a_Eta_0[i_Bs], a_Eta_0[i_Bs]], ax.get_ylim(), 'r--', lw = 2, label = '$N_\mathrm{cl}$=%i'%( dNND['aNND'][dNND['aNND']<1e-5].shape[0]))

            ax.legend( loc = 'upper left')
            ax.set_xlabel( 'NND, log$_{10} \eta$')
            ax.set_ylabel( 'Количество событий')
            ax.grid( 'on')
            ax.set_xlim( dPar['xmin'], dPar['xmax'])

            #==================================4==============================================================
            #                           плотностной график T-R (время-расстояние)
            #=================================================================================================
            # Создаем отдельные каталоги для родительских и дочерних событий
            catChild = EqCat()
            catParent= EqCat()
            catChild.copy(  ranCat)
            catParent.copy( ranCat)

            # Отбираем события по ID родительских и дочерних пар
            catChild.selEventsFromID(    dNND['aEqID_c'], repeats = True)
            catParent.selEventsFromID(   dNND['aEqID_p'], repeats = True)
            print( catChild.size(), catParent.size(), eqCatMc.size())
            
            # Вычисляем масштабированные времена и расстояния
            a_R, a_T = clustering.rescaled_t_r( catChild, catParent, dConst, correct_co_located = True)

            # Создаем сетку для плотностного графика
            a_Tbin = np.arange( dPar['Tmin'], dPar['Tmax']+2*dPar['binx'], dPar['binx'])
            a_Rbin = np.arange( dPar['Rmin'], dPar['Rmax']+2*dPar['biny'], dPar['biny'])
            a_log_T = np.log10( a_T)
            a_log_R = np.log10( a_R)
            
            # Вычисляем двумерную плотность
            XX, YY, ZZ = data_utils.density_2D( a_log_T, a_log_R, a_Tbin, a_Rbin, sigma = dPar['sigma'])

            # Строим плотностной график
            plt.figure(2, figsize= (8,10))
            ax = plt.subplot(111)
            ax.set_title( 'Пары ближайших соседей на графике Время-Расстояние')
            
            # Отображаем плотность цветом
            normZZ = ZZ*( dPar['binx']*dPar['biny']*eqCatMc.size())
            plot1 = ax.pcolormesh( XX, YY, normZZ, cmap=dPar['cmap'])
            cbar  = plt.colorbar(plot1, orientation = 'horizontal', shrink = .5, aspect = 20,)
            
            # Рисуем линию, разделяющую кластерные и фоновые события
            ax.plot( [dPar['Tmin'], dPar['Tmax']],  -np.array([dPar['Tmin'], dPar['Tmax']])+a_Eta_0[i_Bs], '-', lw = 1.5, color = 'w' )
            ax.plot( [dPar['Tmin'], dPar['Tmax']],  -np.array([dPar['Tmin'], dPar['Tmax']])+a_Eta_0[i_Bs],'--', lw = 1.5, color = '.5' )
            
            # Настройка подписей
            cbar.set_label( 'Количество пар событий',labelpad=-40)
            ax.set_xlabel( 'Масштабированное время')
            ax.set_ylabel( 'Масштабированное расстояние')
            ax.set_xlim( dPar['Tmin'], dPar['Tmax'])
            ax.set_ylim( dPar['Rmin'], dPar['Rmax'])

            plt.show()

    f_eta_0 = a_Eta_0.mean()
    print('среднее значение eta_0', f_eta_0)
    
    file_out = f"data/{file_in}_Mc_{f_Mc:.1f}_eta_0.txt"  # Используем f-строку для современного формата
    np.savetxt(file_out, np.array([f_eta_0]), fmt='%10.3f', header='eta_0')
    print('результаты сохранены в', file_out)
    
    scipy.io.savemat(file_out.replace('txt', 'mat'), {'eta_0': f_eta_0, 'eta_BS': a_Eta_0}, do_compression=True)

общее количество событий 13
количество событий после начального отбора 13
-------------- текущий порог Mc: 3.0 ---------------------
размер каталога после отбора по магнитуде 13
среднее значение eta_0 -2.9965830000000007
результаты сохранены в data/NEIC_Global_Chignik_2021.mat_Mc_3.0_eta_0.txt
-------------- текущий порог Mc: 4.0 ---------------------
размер каталога после отбора по магнитуде 13
среднее значение eta_0 -2.9805634999999997
результаты сохранены в data/NEIC_Global_Chignik_2021.mat_Mc_4.0_eta_0.txt
